# Process Hackathons

In [20]:
import pandas as pd
import pathlib
import requests
from tqdm.asyncio import tqdm
import re

In [21]:
data_root = pathlib.Path("../data")
output_root = data_root / pathlib.Path("outputs")
output_files = output_root.glob("project-output*.csv")

df = None
for file in output_files:
    df_partial = pd.read_csv(file)
    if df is None:
        df = df_partial
    else:
        df = pd.concat([df, df_partial], ignore_index=True)

df.head()


,title,description,built-with,is-winner,full-description,hackathon_id
0,NaN,NaN,"twitter,android",False,\n \n Viewing Tweets without their usern...,936
1,NaN,NaN,weebly,False,\n \n https://docs.google.com/presentati...,2401
2,NaN,NaN,NaN,False,\n \n Jewish learning has traditionally ...,2401
3,NaN,NaN,NaN,False,\n \n Inspiration \n\n Elementary school...,2401
4,NaN,NaN,"html,webgl,javascript,web",False,\n \n We like pictures and so originally...,1606


In [22]:
def clean_description(description: str):
    if not isinstance(description, str):
        return ""

    text = str(description).lower()
    text = re.sub(r"http\S+", "", text)
    text = re.sub(r"[^a-z\s]", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

df["description"] = df["description"].apply(clean_description)
df["full-description"] = df["full-description"].apply(clean_description)
df.head()


,title,description,built-with,is-winner,full-description,hackathon_id
0,NaN,,"twitter,android",False,viewing tweets without their username or infor...,936
1,NaN,,weebly,False,,2401
2,NaN,,NaN,False,jewish learning has traditionally been done in...,2401
3,NaN,,NaN,False,inspiration elementary school students receive...,2401
4,NaN,,"html,webgl,javascript,web",False,we like pictures and so originally we wanted t...,1606


In [23]:
df.to_csv(data_root / "projects.csv", index=False)

# Process Hackathons

In [24]:
hackathon_df = pd.read_csv(data_root / "hackathons.csv")
hackathon_df.head()

,id,title,url,submission_period_dates,themes,registrations_count
0,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264
1,16954,Winter MelonJam 2025,https://wmj2025.devpost.com/,"Dec 26 - 29, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",49
2,27408,Digital Innovation Challenge 2025 - Finance Track,https://dic-2025-finance-track.devpost.com/,"Nov 20 - Dec 29, 2025","[{'id': 3, 'name': 'Fintech'}, {'id': 6, 'name...",176
3,27220,Artificathon 2025,https://artificathon-2025.devpost.com/,"Dec 01 - 28, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",56
4,27377,Square Hacks- Build for Rural India,https://squarehacks.devpost.com/,"Dec 26 - 27, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",472


In [25]:
from datetime import datetime

def parse_submission_dates(submission_period_dates: str) -> tuple[datetime | None, datetime | None]:
    if submission_period_dates is None or not isinstance(submission_period_dates, str):
        return None, None

    input_length = len(submission_period_dates)
    if input_length == len("Dec 26, 2025"):
        start_time = datetime.strptime(submission_period_dates, "%b %d, %Y")
        end_time = start_time

    elif input_length == len("Dec 26 - 29, 2025"):
        month = submission_period_dates[:3]
        start_date = submission_period_dates[4:6]
        end_date = submission_period_dates[9:11]
        year = submission_period_dates[-4:]

        start_time = datetime.strptime(f"{month} {start_date}, {year}", "%b %d, %Y")
        end_time = datetime.strptime(f"{month} {end_date}, {year}", "%b %d, %Y")

    elif input_length == len("Nov 20 - Dec 29, 2025"):
        start_date = submission_period_dates.split(" - ")[0]
        end_date = submission_period_dates.split(" - ")[1].split(",")[0]
        year = submission_period_dates[-4:]

        start_time = datetime.strptime(f"{start_date}, {year}", "%b %d, %Y")
        end_time = datetime.strptime(f"{end_date}, {year}", "%b %d, %Y")

    elif input_length == len("Dec 04, 2024 - Jan 24, 2025"):
        start_full_date, end_full_date = submission_period_dates.split(" - ")

        start_time = datetime.strptime(start_full_date, "%b %d, %Y")
        end_time = datetime.strptime(end_full_date, "%b %d, %Y")

    else:
        raise ValueError(f"Unexpected submission period dates {submission_period_dates}")

    return start_time, end_time

# Sanity check
accepted_start = datetime(2008, 11, 1)
accepted_end = datetime(2026, 12, 31)
for idx, row in hackathon_df.iterrows():
    parsed_start, parsed_end = parse_submission_dates(row["submission_period_dates"])
    assert accepted_start <= parsed_start <= parsed_end <= accepted_end

In [26]:
hackathon_df[["submission_start", "submission_end"]] = (
    hackathon_df["submission_period_dates"]
    .apply(parse_submission_dates)
    .apply(pd.Series)
)

hackathon_df.head()

,id,title,url,submission_period_dates,themes,registrations_count,submission_start,submission_end
0,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31
1,16954,Winter MelonJam 2025,https://wmj2025.devpost.com/,"Dec 26 - 29, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",49,2025-12-26,2025-12-29
2,27408,Digital Innovation Challenge 2025 - Finance Track,https://dic-2025-finance-track.devpost.com/,"Nov 20 - Dec 29, 2025","[{'id': 3, 'name': 'Fintech'}, {'id': 6, 'name...",176,2025-11-20,2025-12-29
3,27220,Artificathon 2025,https://artificathon-2025.devpost.com/,"Dec 01 - 28, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",56,2025-12-01,2025-12-28
4,27377,Square Hacks- Build for Rural India,https://squarehacks.devpost.com/,"Dec 26 - 27, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",472,2025-12-26,2025-12-27


## Add coordinates

In [27]:
geo_location_df = pd.read_csv(data_root / "locations.csv")

In [28]:
hackathon_df_with_locations = pd.merge(hackathon_df, geo_location_df, on="id", how="left")
hackathon_df_with_locations.head()

,id,title,url,submission_period_dates,themes,registrations_count,submission_start,submission_end,geo_location
0,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31,Online
1,16954,Winter MelonJam 2025,https://wmj2025.devpost.com/,"Dec 26 - 29, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",49,2025-12-26,2025-12-29,Online
2,27408,Digital Innovation Challenge 2025 - Finance Track,https://dic-2025-finance-track.devpost.com/,"Nov 20 - Dec 29, 2025","[{'id': 3, 'name': 'Fintech'}, {'id': 6, 'name...",176,2025-11-20,2025-12-29,Online
3,27220,Artificathon 2025,https://artificathon-2025.devpost.com/,"Dec 01 - 28, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",56,2025-12-01,2025-12-28,Online
4,27377,Square Hacks- Build for Rural India,https://squarehacks.devpost.com/,"Dec 26 - 27, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",472,2025-12-26,2025-12-27,"IIT Delhi, Indian Institute of Technology Delh..."


In [29]:
from urllib.parse import quote
from dotenv import load_dotenv
import os
load_dotenv()
api_key = os.getenv("GOOGLE_MAPS_API_KEY")

In [30]:
def get_geo_code(geo_location: str):
    if not isinstance(geo_location, str) or geo_location == "Online":
        return None, None

    try:
        request = "https://maps.googleapis.com/maps/api/geocode/json?address={}&key={}".format(
            quote(geo_location), api_key
        )
        response = requests.get(request)
        response.raise_for_status()
        data = response.json()

        result_obj = data["results"][0]

        admin_area_level1 = None
        country_short_name = None
        country_long_name = None
        for component in result_obj.get("address_components", []):
            if "administrative_area_level_1" in component.get("types", []):
                admin_area_level1 = component.get("long_name")
            if "country" in component.get("types", []):
                country_short_name = component.get("short_name")
                country_long_name = component.get("long_name")

        if admin_area_level1 is not None and country_short_name is not None:
            locality = f"{admin_area_level1}, {country_short_name}"
        elif admin_area_level1 is None and country_short_name is not None:
            locality = country_long_name
        elif country_short_name is None and admin_area_level1 is not None:
            locality = admin_area_level1
        else:
            return None, None

        location_dict = result_obj["geometry"]["location"]
        return (location_dict["lat"], location_dict["lng"]), locality

    except Exception as e:
        print(e)
        return None, None

get_geo_code("IIT Delhi, Indian Institute of Technology Delhi, Hauz Khas, New Delhi, Delhi 110016, India")

((28.5410732, 77.1990842), 'Delhi, IN')

In [31]:
coordinate_arr = []
locality_arr = []
for _, row in tqdm(hackathon_df_with_locations.iterrows(), total=len(hackathon_df_with_locations)):
    coordinate, locality = get_geo_code(row["geo_location"])

    coordinate_arr.append(coordinate)
    locality_arr.append(locality)

hackathon_df_with_coordinates = hackathon_df_with_locations.copy()
hackathon_df_with_coordinates["coordinate"] = coordinate_arr
hackathon_df_with_coordinates["locality"] = locality_arr
hackathon_df_with_coordinates.head()


  1%|          | 52/8688 [00:03<03:33, 40.41it/s]

list index out of range


  1%|▏         | 123/8688 [00:08<10:02, 14.21it/s]

list index out of range


  6%|▌         | 507/8688 [00:46<08:41, 15.70it/s]

...

list index out of range


 98%|█████████▊| 8528/8688 [10:50<00:14, 11.19it/s]

list index out of range


100%|██████████| 8688/8688 [10:58<00:00, 13.19it/s]


,id,title,url,submission_period_dates,themes,registrations_count,submission_start,submission_end,geo_location,coordinate,locality
0,27498,AETHRA GLOBAL VIBEATHON 2025,https://aethra-global-vibeathon-2025.devpost.com/,"Nov 14 - Dec 31, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",264,2025-11-14,2025-12-31,Online,None,NaN
1,16954,Winter MelonJam 2025,https://wmj2025.devpost.com/,"Dec 26 - 29, 2025","[{'id': 23, 'name': 'Beginner Friendly'}, {'id...",49,2025-12-26,2025-12-29,Online,None,NaN
2,27408,Digital Innovation Challenge 2025 - Finance Track,https://dic-2025-finance-track.devpost.com/,"Nov 20 - Dec 29, 2025","[{'id': 3, 'name': 'Fintech'}, {'id': 6, 'name...",176,2025-11-20,2025-12-29,Online,None,NaN
3,27220,Artificathon 2025,https://artificathon-2025.devpost.com/,"Dec 01 - 28, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",56,2025-12-01,2025-12-28,Online,None,NaN
4,27377,Square Hacks- Build for Rural India,https://squarehacks.devpost.com/,"Dec 26 - 27, 2025","[{'id': 19, 'name': 'Education'}, {'id': 6, 'n...",472,2025-12-26,2025-12-27,"IIT Delhi, Indian Institute of Technology Delh...","(28.5410732, 77.1990842)","Delhi, IN"


In [32]:
hackathon_df_with_coordinates.to_csv(data_root / "processed_hackathons.csv", index=False)